# End of week 1 exercise

To demonstrate your familiarity with OpenAI API, and also Ollama, build a tool that takes a technical question,  
and responds with an explanation. This is a tool that you will be able to use yourself during the course!

In [1]:
# imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# constants

MODEL_GPT = 'gpt-4o-mini'
MODEL_LLAMA = 'llama3.2'

In [3]:
# set up environment

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
openai = OpenAI()

API key looks good so far


In [4]:
# system prompt:
ai_tutor_system_prompt = '''
You are an AI tutor for helping developers understand programming questions. When the user asks a question, respond with a very brief and simple summary of the answer. Then, construct a more nuanced and in-depth answer, explaining each relevant subtopic. Your response should be in markdown, with codeblocks for use with code fragments as needed to further explain concepts. Please conclude by prompting the user with a follow-on question that might be useful for them to investigate to develop their understanding of this or any of the related concepts presented in your nuanced explanation.
'''

In [5]:
VERBOSE=True

In [10]:
# Get gpt-4o-mini to answer, with streaming
def query_ai_model(system_prompt, user_prompt, endpoint, selected_model):
    #response = openai.chat.completions.create(
    response = endpoint.chat.completions.create(
        model=selected_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
    )
    result = response.choices[0].message.content
    if VERBOSE:
        print(result)
    display(Markdown(result))



In [14]:
def query_ai_model_stream(system_prompt, user_prompt, endpoint, selected_model):
    #response = openai.chat.completions.create(
    stream = endpoint.chat.completions.create(
        model=selected_model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        stream=True
    )
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [11]:
# here is the question; type over this to ask something new
question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""
query_ai_model(ai_tutor_system_prompt, question, openai, MODEL_GPT)

### Summary
This code creates a generator that yields authors from a list of books, filtering out any books that do not have an author.

### In-Depth Explanation

Let's break down the code: 

```python
yield from {book.get("author") for book in books if book.get("author")}
```

#### Components:

1. **Generator Function**: 
   - The use of `yield from` indicates that this line is likely contained within a generator function. Generators are functions that can yield values one at a time and retain state between each yield.

2. **Set Comprehension**:
   - The part `{book.get("author") for book in books if book.get("author")}` is a set comprehension. It constructs a set, which is an unordered collection of unique elements in Python.
   - The comprehension iterates over an iterable called `books`, which is expected to be a list or similar collection of dictionaries.

3. **Accessing Dictionary Elements**:
   - For each `book` in `books`, `book.get("author")` is called to retrieve the author o

### Summary
This code creates a generator that yields authors from a list of books, filtering out any books that do not have an author.

### In-Depth Explanation

Let's break down the code: 

```python
yield from {book.get("author") for book in books if book.get("author")}
```

#### Components:

1. **Generator Function**: 
   - The use of `yield from` indicates that this line is likely contained within a generator function. Generators are functions that can yield values one at a time and retain state between each yield.

2. **Set Comprehension**:
   - The part `{book.get("author") for book in books if book.get("author")}` is a set comprehension. It constructs a set, which is an unordered collection of unique elements in Python.
   - The comprehension iterates over an iterable called `books`, which is expected to be a list or similar collection of dictionaries.

3. **Accessing Dictionary Elements**:
   - For each `book` in `books`, `book.get("author")` is called to retrieve the author of the book. 
   - If an author is not present (i.e., the key "author" does not exist in the dictionary), `book.get("author")` will return `None`.

4. **Filtering**:
   - The `if book.get("author")` clause filters out any books that do not have an author. As a result, only books with a valid author will be included in the final set.

5. **Yielding Unique Authors**:
   - Lastly, `yield from` will yield each author from the set of authors created by the comprehension. This means that authors will be returned one by one in a generator fashion, and because they are stored in a set, each author will be unique.

### Example Use Case
Here’s an example of how the code could function within a larger context:

```python
def unique_authors(books):
    yield from {book.get("author") for book in books if book.get("author")}

# Example data
books = [
    {"title": "Book 1", "author": "Author A"},
    {"title": "Book 2", "author": "Author B"},
    {"title": "Book 3", "author": None},
    {"title": "Book 4"},
    {"title": "Book 5", "author": "Author A"}
]

# Using the generator
for author in unique_authors(books):
    print(author)
```

This code would print:
```
Author A
Author B
```

### Follow-Up Question
What are the advantages and disadvantages of using sets in this context compared to lists or other collections when you want to ensure all authors are unique?

In [12]:
# Get Llama 3.2 to answer

#we need the local ollama endpoint to be enabled:

ollama = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

ollama_model = "llama3.2:1b"

question = """
Please explain what this code does and why:
yield from {book.get("author") for book in books if book.get("author")}
"""
query_ai_model(ai_tutor_system_prompt, question, ollama, ollama_model)

Here's a brief summary of what this code does:

This code uses a `yield from` statement to extract the authors of certain books from a list of books. It does this by iterating over the list of books, filtering out books without authors, and then yielding the authors.

Here's a more detailed, in-depth explanation:

1. `yield from`: A `yield from` statement is a type of generators in Python that allows you to "return iterators" from a function that returns an iterator. In this case, we're using it to capture the output of another generator expression.

2. `{book.get("author") for book in books`: This is a generator expression that iterates over the list of books. It gets the author of each book using the `get("author")` method and yields the author as the value of each variable `book` in the iterator.

3. `if book.get("author")`: This line filters out books with no authors. It uses the `get("author")` method to check if the author key is present in the current book. The `"author"` key is

Here's a brief summary of what this code does:

This code uses a `yield from` statement to extract the authors of certain books from a list of books. It does this by iterating over the list of books, filtering out books without authors, and then yielding the authors.

Here's a more detailed, in-depth explanation:

1. `yield from`: A `yield from` statement is a type of generators in Python that allows you to "return iterators" from a function that returns an iterator. In this case, we're using it to capture the output of another generator expression.

2. `{book.get("author") for book in books`: This is a generator expression that iterates over the list of books. It gets the author of each book using the `get("author")` method and yields the author as the value of each variable `book` in the iterator.

3. `if book.get("author")`: This line filters out books with no authors. It uses the `get("author")` method to check if the author key is present in the current book. The `"author"` key is the key under which you typically store authors, but it may vary depending on the specific book or library you're working with.

Here's a code snippet that simplifies and expands upon the explanation above:
```python
def get_book_authors(books):
    # Get all the authors of books without their names
    authors = {}
    for book in books:
        # Get the author of the book
        author = book.get("author")
        # Check if the author exists
        if author:
            # Yield the author, which is a substring of the book name
            yield author[0]

    # Yield everything else in the author name
    for k, v in authors.items():
        yield from [u for u, e in v.items() if not k.startswith("a")]

# Define a list of books
books = [
    {"id": 1, "title": "Book1", "author": "John"},
    {"id": 2, "title": "Book2", "author": "Alice"},
    {"id": 3, "title": "Book3", "author": None},
    {"id": 4, "title": "Book4", "author": "Bob"}
]

# Call the function and print the authors
for author in get_book_authors(books):
    print(author)
```

In [15]:
query_ai_model_stream(ai_tutor_system_prompt, question, ollama, ollama_model)

### Code Explanation

This code uses the `yield from` statement in Python. It's designed to filter a list of dictionaries (`books`) based on specific keys and return only the values for those keys.

Here's a breakdown:

- `yield from`: This is the syntax introduced in Python 3.8. It allows the execution of another expression to be unwound and yielded, which in this case will return a result.

- `{ ... }`: This is the dictionary-like syntax that represents a dictionary object without the `dict()` function.

- `book.get("author")`: This gets the value associated with the key `"author"` in the dictionary `book`. If the key is not present, it returns `None`.

- `for book in books`: This loop iterates over each dictionary `book` in the `books` list. If the dictionary `book` has the key `"author"`, it yields the value associated with that key.

### Result

The resulting values are returned as a generator expression, which is an iterator. This allows you to process the generator without storing all the values in memory at once.

### Example

```
>>> books = [
...     {"id": 1, "title": "Book1", "author": "Author1"},
...     {"id": 2, "title": "Book2", "author": "Author2"},
...     {"id": 3, "title": "Book3", "author": "Author3"}
... ]

>>> list(yield from {book.get("author") for book in books if book.get("author")})
[Author1, Author2, Author3]

```

### Why use `yield from` here?

In this case, `yield from` is used to filter out books without an `"author"` key and reduce the number of dictionary lookups required. If you had a very large list of books without any `"author"` key, this can significantly improve performance. However, if the list of books does not contain many `"author"` keys, then this optimization is beneficial.